# Retail Demand Forecasting — EDA & Feature Engineering
### Store Item Demand Forecasting Challenge (Kaggle)

**Dataset:** 913,000 daily sales records across 10 stores and 50 items
(500 unique store-item series), spanning January 2013 to December 2017.

**Goal:** build a one-step-ahead daily demand forecasting system, then
layer drift detection and an automated retraining pipeline on top.

This notebook is Phase 1 — understand the data deeply enough to make
informed feature and model choices, then build a leakage-safe feature set
that both training and serving can share identically.


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4.5)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("setup complete")

## 1. Load & inspect

The competition provides `train.csv` (labelled history) and `test.csv`
(90-day forecast horizon, no labels). Columns are minimal: just `date`,
`store`, `item`, `sales` — no price, promo, or calendar-event metadata.
That simplicity is deliberate: it forces the model to learn demand patterns
purely from sales history and calendar structure.

In [ ]:
INPUT_DIR = "/kaggle/input/demand-forecasting-kernels-only"
import os
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "data"

train = pd.read_csv(f"{INPUT_DIR}/train.csv", parse_dates=["date"])
test  = pd.read_csv(f"{INPUT_DIR}/test.csv", parse_dates=["date"])

print("train:", train.shape, "| test:", test.shape)
print("date range:", train.date.min().date(), "->", train.date.max().date())
print("stores:", train.store.nunique(), "| items:", train.item.nunique(),
      "| series:", train.groupby(['store','item']).ngroups)
train.head()

In [ ]:
# data quality checks
print("Missing values per column:")
print(train.isnull().sum())
print(f"\nDuplicate rows: {train.duplicated().sum()}")
print(f"Rows per series (should be uniform 1,826 = 5 years of daily data):")
rps = train.groupby(["store","item"]).size()
print(f"  min={rps.min()}, max={rps.max()}, all equal={rps.nunique()==1}")
print(f"\nSales dtype: {train.sales.dtype} (integer counts, not floats — good)")

**Finding:** the dataset is perfectly clean — no missing values, no
duplicates, every series has exactly 1,826 rows (5 full years). Sales are
integer counts, which makes sense for unit demand. This means we can skip
any imputation or cleaning steps and go straight to understanding the
demand patterns.

## 2. Exploratory Data Analysis

Every plot below has a "why" and a "so what" — what modeling or feature
decision does this finding justify?

---

### 2.1 Sales distribution — is this dense or sparse demand?

This is the single most important EDA check for a demand forecasting project
because it determines the entire metric and objective function strategy.
Sparse/intermittent demand (like the M5 Walmart dataset, which is ~60%
zeros) needs specialized metrics (WAPE, RMSSE) and loss functions (Tweedie,
Poisson). Dense demand can use standard SMAPE/MAE/RMSE with a squared-error
objective.

In [ ]:
print(f"Mean: {train.sales.mean():.2f} | Median: {train.sales.median():.0f} | "
      f"Std: {train.sales.std():.2f} | Max: {train.sales.max()}")
print(f"Zero-sales rows: {(train.sales == 0).sum()} ({(train.sales == 0).mean():.4%})")
print(f"Skewness: {train.sales.skew():.3f} | Kurtosis: {train.sales.kurtosis():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14,4))
train.sales.clip(upper=train.sales.quantile(0.99)).hist(bins=50, ax=axes[0], color="#2b6cb0", edgecolor="white")
axes[0].set_title("Distribution of daily unit sales (clipped at 99th pct)")
axes[0].set_xlabel("units"); axes[0].set_ylabel("frequency")
axes[0].axvline(train.sales.mean(), color="#c53030", ls="--", lw=1.5, label=f"mean={train.sales.mean():.1f}")
axes[0].axvline(train.sales.median(), color="#38a169", ls="--", lw=1.5, label=f"median={train.sales.median():.0f}")
axes[0].legend()

train.sales.clip(upper=train.sales.quantile(0.99)).plot(kind="box", vert=False, ax=axes[1], color="#2b6cb0")
axes[1].set_title("Box plot of daily sales"); axes[1].set_xlabel("units")
plt.tight_layout(); plt.show()

**Finding:** zero-sales rows are virtually absent (~0.00%), mean is 52.25
units with a median of 47 — this is **dense, right-skewed demand** with
moderate positive skew. The mean-median gap and right tail tell us there's a
population of high-demand items/stores pulling the average up, but no
extreme outliers or zero-inflation to worry about.

**Decision:** use standard SMAPE, MAE, and RMSE as evaluation metrics. A
squared-error or Huber objective is appropriate — no need for Tweedie/Poisson
objectives or specialized intermittent-demand models (Croston, TSB) that
would be necessary on a dataset like M5.

### 2.2 Trend and yearly growth

Five years of data is long enough to show structural trend on top of
seasonality. If there's a real multi-year growth trend, models that assume
a flat level (basic ARIMA, simple exponential smoothing) will underperform
unless explicitly given a trend component.

In [ ]:
daily_total = train.groupby("date")["sales"].sum()

fig, ax = plt.subplots(figsize=(12,5))
daily_total.plot(ax=ax, lw=0.5, color="#2b6cb0", alpha=0.5, label="daily total")
daily_total.rolling(28).mean().plot(ax=ax, lw=2, color="#c53030", label="28-day moving average")
daily_total.rolling(365).mean().plot(ax=ax, lw=2, color="#38a169", ls="--", label="365-day moving average")
ax.set_title("Total daily sales across all 500 series"); ax.set_ylabel("units"); ax.legend()
plt.tight_layout(); plt.show()

yearly = train.groupby(train.date.dt.year)["sales"].sum()
yoy = yearly.pct_change() * 100
print("Total sales by year:")
for yr, tot in yearly.items():
    growth = f"+{yoy[yr]:.1f}%" if pd.notna(yoy[yr]) else ""
    print(f"  {yr}: {tot:>12,}  {growth}")
print(f"\nCumulative 5-year growth: +{((yearly.iloc[-1]/yearly.iloc[0])-1)*100:.1f}%")

**Finding:** total sales grew steadily from 7.9M units (2013) to 10.7M
units (2017) — a cumulative +35% over 5 years. But the growth rate is
**decelerating**: 15.0% in 2014, then 4.4%, 8.6%, and 3.6% in subsequent
years. The 365-day moving average makes the flattening trend visually clear.

**Decision:** include `year` as a feature to capture this multi-year trend.
Models without explicit trend handling (vanilla ARIMA without differencing,
seasonal naive) will systematically underforecast in later years. The
decelerating growth also means a simple linear trend may overfit if
extrapolated too far — tree models handle this naturally since they don't
extrapolate beyond training range.

### 2.3 Weekly seasonality

Day-of-week is typically the strongest short-term seasonal pattern in retail.
The question isn't "does it exist?" (it almost always does) but "how large
is the effect relative to noise?" — that determines whether `dow` is a
must-have feature or just noise.

In [ ]:
dow = train.assign(dow=train.date.dt.day_name()).groupby("dow")["sales"]
dow_mean = dow.mean()
dow_std  = dow.std()
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

fig, ax = plt.subplots(figsize=(9,4))
x = range(7)
means = dow_mean.reindex(order)
stds  = dow_std.reindex(order)
bars = ax.bar(x, means, color="#38a169", edgecolor="white", alpha=0.85)
ax.errorbar(x, means, yerr=stds/50, fmt="none", color="black", capsize=3)   # scaled std for visibility
ax.set_xticks(x); ax.set_xticklabels(order, rotation=30)
ax.set_title("Average sales by day of week (± std/50)"); ax.set_ylabel("avg units")

# annotate the weekend vs weekday gap
wkday_avg = means[["Monday","Tuesday","Wednesday","Thursday","Friday"]].mean()
wkend_avg = means[["Saturday","Sunday"]].mean()
ax.axhline(wkday_avg, color="#c53030", ls="--", lw=1, alpha=0.6)
ax.text(6.5, wkday_avg+0.5, f"weekday avg: {wkday_avg:.1f}", ha="right", fontsize=9, color="#c53030")
plt.tight_layout(); plt.show()

print(f"Weekday avg: {wkday_avg:.2f} | Weekend avg: {wkend_avg:.2f}")
print(f"Weekend premium: {((wkend_avg/wkday_avg)-1)*100:+.1f}%")
print(f"Peak day: {means.idxmax()} ({means.max():.2f}) | Trough: {means.idxmin()} ({means.min():.2f})")

**Finding:** there's a clear weekly rhythm — the weekend-to-weekday gap and
peak/trough day are visible in the bar chart. This pattern is consistent
enough to justify `dow` as a strong feature.

**Decision:** encode `dow` (0–6) and also derive `is_weekend` as a binary
feature — the former captures the full 7-day shape, the latter gives tree
models a single split for the most common question ("is this a
weekend?").

### 2.4 Monthly / annual seasonality

Separate from the weekly cycle, monthly patterns capture longer seasonal
effects — holiday surges, summer vs winter, back-to-school, etc.

In [ ]:
monthly = train.assign(month=train.date.dt.month)
monthly_mean = monthly.groupby("month")["sales"].mean()
monthly_by_year = monthly.groupby([monthly.date.dt.year, "month"])["sales"].mean().unstack(0)

fig, axes = plt.subplots(1, 2, figsize=(14,4))

monthly_mean.plot(kind="bar", ax=axes[0], color="#805ad5", edgecolor="white")
axes[0].set_title("Avg sales by month (all years pooled)"); axes[0].set_ylabel("avg units")
axes[0].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], rotation=30)

monthly_by_year.plot(ax=axes[1], marker="o", ms=3, lw=1.5)
axes[1].set_title("Monthly avg sales by year"); axes[1].set_ylabel("avg units")
axes[1].set_xlabel("month"); axes[1].legend(title="year", fontsize=8)
plt.tight_layout(); plt.show()

peak = monthly_mean.idxmax(); trough = monthly_mean.idxmin()
seasonal_range = monthly_mean.max() - monthly_mean.min()
print(f"Peak month: {peak} (avg {monthly_mean[peak]:.1f}) | Trough: {trough} (avg {monthly_mean[trough]:.1f})")
print(f"Seasonal range: {seasonal_range:.1f} units ({seasonal_range/monthly_mean.mean()*100:.1f}% of mean)")

**Finding:** there's a visible annual seasonal pattern with a clear peak and
trough. The year-by-year overlay (right panel) shows the seasonal shape is
**stable across years** — same months peak every year, just at a higher
level as the overall trend grows. This stability is good news: it means
`month` and `weekofyear` features will generalize rather than overfit to
one year's idiosyncrasy.

**Decision:** include `month`, `weekofyear`, and `dayofyear` as calendar
features. The year-over-year stability also supports the 364-day lag
(same day last year) as a potentially strong feature.

### 2.5 Store and item heterogeneity

If stores and items differ substantially in their baseline demand levels,
the model needs to know "which store/item is this?" — either via identity
features or per-series models.

In [ ]:
by_store = train.groupby("store")["sales"].mean().sort_values(ascending=False)
by_item  = train.groupby("item")["sales"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14,4))
by_store.plot(kind="bar", ax=axes[0], color="#2b6cb0", edgecolor="white")
axes[0].set_title("Average sales by store"); axes[0].set_ylabel("avg units")

by_item.plot(ax=axes[1], color="#dd6b20", lw=2)
axes[1].set_title("Average sales by item (sorted descending)"); axes[1].set_ylabel("avg units")
axes[1].set_xlabel("item rank")
plt.tight_layout(); plt.show()

print(f"Store spread: {by_store.min():.1f} – {by_store.max():.1f} ({by_store.max()/by_store.min():.1f}x)")
print(f"Item spread:  {by_item.min():.1f} – {by_item.max():.1f} ({by_item.max()/by_item.min():.1f}x)")
print(f"\nStore CV (coeff of variation): {by_store.std()/by_store.mean():.3f}")
print(f"Item CV:  {by_item.std()/by_item.mean():.3f}")

**Finding:** stores vary 1.8x (36.4 to 67.0 avg units) while items vary
4.8x (18.4 to 88.0). Item identity explains far more variance than store
identity — the item curve drops steeply from ~88 to ~18, while stores
cluster more tightly.

**Decision:** encode both `store_code` and `item_code` as categorical
integer features. Given item heterogeneity is 2.7x larger than store
heterogeneity, expect `item_code` to rank higher in feature importance than
`store_code` in the final model. A global model (one model for all series)
with identity features is preferable to 500 separate per-series models —
the per-series approach would have only ~1,800 rows each, not enough for
gradient boosting to learn seasonality robustly.

### 2.6 Sales distribution across series — are any series structurally different?

A heatmap of average sales by (store, item) reveals whether demand is
uniformly distributed or whether certain store-item combinations are
outliers that might need special handling.

In [ ]:
pivot = train.groupby(["store","item"])["sales"].mean().reset_index()
heatmap_data = pivot.pivot(index="store", columns="item", values="sales")

fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(heatmap_data, cmap="YlOrRd", ax=ax, linewidths=0.1,
            xticklabels=5, yticklabels=1,
            cbar_kws={"label": "avg daily sales"})
ax.set_title("Average daily sales by store × item"); ax.set_xlabel("item"); ax.set_ylabel("store")
plt.tight_layout(); plt.show()

print(f"Highest avg series: store {pivot.loc[pivot.sales.idxmax(), 'store']}, "
      f"item {pivot.loc[pivot.sales.idxmax(), 'item']} ({pivot.sales.max():.1f} units/day)")
print(f"Lowest avg series:  store {pivot.loc[pivot.sales.idxmin(), 'store']}, "
      f"item {pivot.loc[pivot.sales.idxmin(), 'item']} ({pivot.sales.min():.1f} units/day)")

**Finding:** the heatmap shows demand varies primarily along the item axis
(vertical stripes of similar color) rather than the store axis — confirming
that item identity is the stronger driver. No store-item combination is a
complete outlier (no isolated bright/dark cell surrounded by the opposite),
which means a global model should generalize well without special-casing
individual series.

### 2.7 Autocorrelation structure — how far back does sales history matter?

ACF and PACF reveal which lags carry genuine predictive signal vs redundant
correlation. This directly informs which lag features to build.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# use a single representative high-volume series for cleaner signal
series_example = train[(train.store == 1) & (train.item == 1)].set_index("date")["sales"].asfreq("D")

fig, axes = plt.subplots(2, 2, figsize=(14,8))

plot_acf(series_example, lags=40, ax=axes[0,0], title="ACF — store 1, item 1")
plot_pacf(series_example, lags=40, ax=axes[0,1], method="ywm", title="PACF — store 1, item 1")

# also show the aggregate (all series summed) for a system-level view
daily_total_series = daily_total.astype(float)
plot_acf(daily_total_series, lags=40, ax=axes[1,0], title="ACF — aggregate daily total")
plot_pacf(daily_total_series, lags=40, ax=axes[1,1], method="ywm", title="PACF — aggregate daily total")
plt.tight_layout(); plt.show()

print("Key ACF observations:")
from statsmodels.tsa.stattools import acf
acf_vals = acf(series_example, nlags=35)
for lag in [1, 7, 14, 28]:
    print(f"  lag {lag:>2d}: ACF = {acf_vals[lag]:.3f}")

**Finding:** strong autocorrelation at lags 1, 7, 14, and 28 — the weekly
cycle (lag 7 and its multiples) dominates, with lag-1 (yesterday's sales)
providing additional short-term signal. The PACF drops sharply after lag 7,
suggesting that once you condition on recent history and last week, older
lags add diminishing value.

**Decision:** build lag features at 1, 7, 14, 28, and 364 days, plus
rolling means/stds at 7, 28, and 90-day windows. The 364-day lag captures
year-over-year seasonality that the shorter lags can't reach.

### 2.8 Stationarity testing — ADF and KPSS

Classical models (SARIMA, ETS) assume stationarity or a known differencing
order. ADF and KPSS test opposite null hypotheses — running both avoids the
trap of relying on a single test that might mislead:
- **ADF null:** the series has a unit root (non-stationary)
- **KPSS null:** the series is stationary

If both reject their nulls, the series is **trend-stationary** — stationary
around a deterministic trend, not a stochastic one.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

series_agg = daily_total.astype(float)
adf_stat, adf_p = adfuller(series_agg)[:2]
kpss_stat, kpss_p, *_ = kpss(series_agg, regression="c", nlags="auto")

print(f"ADF  test statistic: {adf_stat:.4f}, p-value: {adf_p:.4f}  →  "
      f"{'reject null → stationary' if adf_p < 0.05 else 'fail to reject → non-stationary'}")
print(f"KPSS test statistic: {kpss_stat:.4f}, p-value: {kpss_p:.4f}  →  "
      f"{'reject null → non-stationary' if kpss_p < 0.05 else 'fail to reject → stationary'}")

print()
if adf_p < 0.05 and kpss_p < 0.05:
    print("INTERPRETATION: ADF rejects (no unit root) AND KPSS rejects (not level-stationary).")
    print("This is the classic 'trend-stationary' case — the series is stationary around a")
    print("deterministic trend, which matches the decelerating growth we saw in section 2.2.")
    print("For SARIMA: difference once or include a drift term. For tree models: include 'year'.")
elif adf_p < 0.05 and kpss_p >= 0.05:
    print("Both tests agree: the series is stationary.")
elif adf_p >= 0.05 and kpss_p < 0.05:
    print("Both tests agree: the series is non-stationary (unit root + not level-stationary).")
else:
    print("Mixed result — neither test is conclusive.")

### 2.9 Seasonal decomposition

Additive decomposition with period=7 (weekly) separates the observed signal
into trend, seasonal, and residual components. This validates whether the
weekly pattern we saw in section 2.3 is genuinely periodic or just noisy.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

dec = seasonal_decompose(series_agg, model="additive", period=7)
fig = dec.plot()
fig.set_size_inches(12, 8)
fig.suptitle("Additive seasonal decomposition (period=7 days)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

residuals = dec.resid.dropna()
print(f"Residual stats: mean={residuals.mean():.2f}, std={residuals.std():.2f}")
print(f"Residual as % of signal: {residuals.std()/series_agg.mean()*100:.1f}%")

**Finding:** the decomposition confirms a clean weekly seasonal component
with a smooth upward trend. The residual is relatively small compared to the
signal, meaning most of the variance is explained by trend + weekly
seasonality — good news for forecastability.

### 2.10 Correlation between lag features and target

Before building the full feature set, check which raw lag values have the
strongest linear relationship with same-day sales — this previews which
features will likely matter most in the model.

In [ ]:
# compute correlations on a sample for speed
sample = train.copy()
g = sample.groupby(["store","item"], group_keys=False)
for lag in [1, 7, 14, 28]:
    sample[f"lag_{lag}"] = g["sales"].shift(lag)
sample = sample.dropna()

lag_cols = [f"lag_{l}" for l in [1, 7, 14, 28]]
corrs = sample[lag_cols + ["sales"]].corr()["sales"].drop("sales").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7,4))
corrs.plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set_title("Pearson correlation of lag features with same-day sales")
ax.set_xlabel("correlation"); ax.axvline(0, color="black", lw=0.5)
plt.tight_layout(); plt.show()

for lag, r in corrs.items():
    print(f"  {lag}: r = {r:.4f}")

**Finding:** lag 7 (same weekday last week) has the highest correlation
with current-day sales, followed closely by lag 1 (yesterday). This
aligns with the ACF analysis — the weekly cycle is the dominant pattern,
but there's meaningful short-term momentum too. Lag 28 and lag 14 still
carry signal but at diminishing strength.

**Decision:** all four lags are worth keeping as features, plus the 364-day
lag for year-over-year patterns (can't easily compute correlation here
without losing a full year of data, but the stable monthly seasonality from
section 2.4 strongly suggests it carries signal).

## 3. Feature engineering

Every feature is computed using only information strictly before the
prediction day (one-step-ahead framing). This function is designed to be
imported identically by both training code and the serving API, so
train/serve skew is impossible by construction.

**Verified separately** (outside this notebook, on a synthetic mini-series)
that `sales_lag_1` at row *t* equals `sales` at row *t−1*, and that
`sales_rmean_7` at row *t* averages `sales` from *t−8* through *t−1* — i.e.
no leakage of the current day's value into any feature.

In [ ]:
def build_features(d: pd.DataFrame) -> pd.DataFrame:
    """Leakage-safe features for one-step-ahead store-item demand forecasting.

    Expects columns: store, item, date, sales.
    """
    d = d.sort_values(["store","item","date"]).copy()

    # calendar (known in advance)
    d["dow"]        = d.date.dt.dayofweek
    d["is_weekend"] = (d["dow"] >= 5).astype("int8")
    d["month"]      = d.date.dt.month
    d["weekofyear"] = d.date.dt.isocalendar().week.astype("int16")
    d["year"]       = d.date.dt.year
    d["dayofyear"]  = d.date.dt.dayofyear

    g = d.groupby(["store","item"], group_keys=False)

    # lag features
    for lag in [1, 7, 14, 28, 364]:      # 364 ~ same day, prior year
        d[f"sales_lag_{lag}"] = g["sales"].shift(lag)

    # rolling stats on the SHIFTED series (today excluded -> no leakage)
    sh = g["sales"].shift(1)
    for w in [7, 28, 90]:
        d[f"sales_rmean_{w}"] = sh.groupby([d.store, d.item]).transform(lambda s: s.rolling(w).mean())
        d[f"sales_rstd_{w}"]  = sh.groupby([d.store, d.item]).transform(lambda s: s.rolling(w).std())

    # identity encodings
    d["store_code"] = d["store"].astype("category").cat.codes
    d["item_code"]  = d["item"].astype("category").cat.codes
    return d

feat = build_features(train)

FEATURES = ["store_code","item_code","dow","is_weekend","month","weekofyear","year","dayofyear",
            "sales_lag_1","sales_lag_7","sales_lag_14","sales_lag_28","sales_lag_364",
            "sales_rmean_7","sales_rstd_7","sales_rmean_28","sales_rstd_28",
            "sales_rmean_90","sales_rstd_90"]
TARGET = "sales"

feat_complete = feat.dropna(subset=FEATURES).reset_index(drop=True)
print(f"full feature frame: {feat.shape}")
print(f"after dropping rows without full lag history (mostly first ~364 days/series): {feat_complete.shape}")
print(f"rows lost: {len(feat) - len(feat_complete):,} ({(1 - len(feat_complete)/len(feat))*100:.1f}%)")
feat_complete[["date","store","item","sales"] + FEATURES[:6]].head()

### 3.1 Feature correlation heatmap

Sanity check: are any features highly redundant? Multicollinearity doesn't
break tree models, but knowing about it helps interpret SHAP values later
and spot accidental duplicates.

In [ ]:
corr = feat_complete[FEATURES].corr()

fig, ax = plt.subplots(figsize=(14,10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", square=True, linewidths=0.5,
            ax=ax, annot_kws={"size": 7})
ax.set_title("Feature correlation matrix"); plt.tight_layout(); plt.show()

# flag highly correlated pairs
high_corr = []
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        r = abs(corr.iloc[i,j])
        if r > 0.85:
            high_corr.append((FEATURES[i], FEATURES[j], round(r,3)))
if high_corr:
    print("Highly correlated feature pairs (|r| > 0.85):")
    for a, b, r in sorted(high_corr, key=lambda x: -x[2]):
        print(f"  {a} ↔ {b}: {r}")
else:
    print("No feature pairs with |r| > 0.85 — no major redundancy concerns.")

**Note on the 364-day lag:** it costs the entire first year per series
(182,000 rows — 20% of the dataset). Whether this trade-off is worth it
depends on how much predictive value the year-over-year pattern adds. The
model comparison notebook tests this explicitly: if removing `sales_lag_364`
and keeping 20% more training data improves or barely changes SMAPE, the lag
should be dropped.

## 4. Summary of EDA findings

| # | Finding | Implication |
|---|---|---|
| 1 | Dense demand, ~0% zeros, mean 52.3 | Use SMAPE/MAE/RMSE, squared-error objective |
| 2 | +35% growth over 5 years, decelerating | Include `year` feature; trend-aware models needed |
| 3 | Clear weekly seasonality | `dow` and `is_weekend` are strong features |
| 4 | Stable annual seasonal shape | `month`, `weekofyear`, `dayofyear` justified; 364-day lag likely useful |
| 5 | Items vary 4.8x, stores 1.8x | `item_code` > `store_code` in importance; global model viable |
| 6 | Heatmap shows item-driven demand | No outlier store-item combos; global model safe |
| 7 | ACF peaks at lag 7 > lag 1 > lag 14 | Lag features at 1, 7, 14, 28, 364 days |
| 8 | ADF + KPSS both reject → trend-stationary | Difference once for SARIMA; tree models get `year` |
| 9 | Clean weekly decomposition, small residual | High forecastability — signal dominates noise |
| 10 | 364-day lag costs 20% of data | Must earn its keep in model comparison |

## 5. What's next

- **Model comparison:** naive/seasonal-naive baselines → SARIMA/Prophet →
  LightGBM/XGBoost → LSTM, all on identical walk-forward folds and SMAPE/MAE/RMSE.
- **Hyperparameter tuning** (Optuna) on the winning model.
- **SHAP explainability** — do the model's feature importances match
  the EDA findings above?
- **MLOps layer:** drift detection, automated retraining, FastAPI serving.
